## Weather Agent

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

In [ ]:
from typing import TypedDict
from langchain.agents import create_agent
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    city: str
    weather: str

llm = ChatGroq(model = "llama-3.3-70b-versatile")

@tool
def get_weather(state: State = None, city: str = None) -> dict:
    """Get the weather forecast/details for a given city.
    
    Args:
        state: A dictionary containing the target 'city' (e.g. {"city": "Berlin"}).
        city: The name of the city (e.g. "Berlin").
    """
    import requests
    
    city_name = city
    if not city_name and state:
        if isinstance(state, str):
            city_name = state
        elif isinstance(state, dict):
            city_name = state.get("city")
            
    if not city_name:
        return {"error": "City parameter is missing."}
        
    # Geocode city to coordinates
    geocode_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}&count=1&language=en&format=json"
    try:
        geo_res = requests.get(geocode_url).json()
        results = geo_res.get("results")
        if not results:
            return {"error": f"City '{city_name}' not found."}
        lat = results[0]["latitude"]
        lon = results[0]["longitude"]
    except Exception as e:
        return {"error": f"Geocoding failed for '{city_name}': {str(e)}"}
        
    # Fetch weather forecast (1 day only to keep response small)
    weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=temperature_2m&forecast_days=1"
    try:
        weather_res = requests.get(weather_url).json()
        return weather_res
    except Exception as e:
        return {"error": f"Weather lookup failed for coordinates ({lat}, {lon}): {str(e)}"}

system_prompt = "You are a weather agent. Answer appropriately."

agent = create_agent(
    model=llm,
    tools = [get_weather],
    system_prompt=system_prompt,
    checkpointer=MemorySaver()
    )

In [19]:
import uuid

# Invoke the agent to get the weather for a city
inputs = {"messages": [{"role": "user", "content": "What's the weather in plantation right now?"}]}

# Fresh thread_id each run so MemorySaver doesn't accumulate old tool responses
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Streaming agent response:\n")
final_response = None
for event in agent.stream(inputs, config=config, stream_mode="values"):
    if "messages" in event:
        msg = event["messages"][-1]
        if msg.type == "ai" and msg.content:
            final_response = msg

if final_response:
    final_response.pretty_print()

Streaming agent response:

================================== Ai Message ==================================

The current weather in Plantation is 29.5°C.
